# 03 — Quintile Portfolio Sorts

Constructs long/short VRP quintile portfolios and evaluates performance.

- Value-weighted within quintiles
- Monthly rebalancing
- Evaluates raw return, CAPM α, and net-of-cost return (5bps round-trip)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')
from src.portfolio import quintile_sort, summary_statistics, capm_alpha

DATA_DIR = '../data'
vrp = pd.read_parquet(f'{DATA_DIR}/vrp_panel.parquet')
returns = pd.read_parquet(f'{DATA_DIR}/returns_panel.parquet')
market_cap = pd.read_parquet(f'{DATA_DIR}/market_cap_panel.parquet')

ModuleNotFoundError: No module named 'matplotlib'

## Full-period quintile sort

In [ ]:
port_ret = quintile_sort(
    signal=vrp,
    returns=returns,
    n_portfolios=5,
    weighting='value_weighted',
    market_cap=market_cap,
    transaction_cost_bps=5.0,
)

stats = summary_statistics(port_ret, freq=12)
print('=== Portfolio summary (annualised) ===')
print(stats[['ann_return', 'ann_vol', 'sharpe', 'max_drawdown']].round(4))

## Cumulative returns — full history

In [2]:
cum = (1 + port_ret[['Q1', 'Q5', 'LS', 'LS_net']]).cumprod()

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(cum.index, cum['LS'],     label='L/S gross',    color='#185FA5', linewidth=1.8)
ax.plot(cum.index, cum['LS_net'], label='L/S net (5bps)',color='#BA7517', linewidth=1.4, linestyle='--')
ax.plot(cum.index, cum['Q5'],     label='Q5 (high VRP)',color='#639922', linewidth=1.2, alpha=0.8)
ax.plot(cum.index, cum['Q1'],     label='Q1 (low VRP)', color='#E24B4A', linewidth=1.2, alpha=0.8)
ax.axhline(1, color='#888780', linewidth=0.6, linestyle=':')
ax.set_title('VRP quintile portfolios — cumulative returns', fontsize=12)
ax.set_ylabel('Growth of $1', fontsize=11)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../data/cumulative_returns.png', dpi=150, bbox_inches='tight')
plt.show()

NameError: name 'port_ret' is not defined

## CAPM alpha per sub-period

In [3]:
mkt = returns.mean(axis=1)
ls = port_ret['LS'].dropna()

sub_periods = [
    ('1996-2009', '2000-01-01', '2009-12-31'),
    ('2010-2015', '2010-01-01', '2015-12-31'),
    ('2016-2022', '2016-01-01', '2022-12-31'),
    ('2023-2025', '2023-01-01', '2025-12-31'),
]

alpha_rows = []
for label, start, end in sub_periods:
    mask = (ls.index >= start) & (ls.index <= end)
    if mask.sum() < 12:
        continue
    res = capm_alpha(ls[mask], mkt[mask], annualize=True)
    alpha_rows.append({'period': label, **res})

alpha_df = pd.DataFrame(alpha_rows).set_index('period')
print('=== CAPM alpha (annualised) ===')
print(alpha_df[['alpha', 'beta', 't_alpha', 'r_squared']].round(4))

NameError: name 'returns' is not defined